#1. Install the Local Engine

In [1]:
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install -q kokoro>=0.9.4 soundfile torch

#2. Connect to Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#3. The Kokoro Generator Script

In [3]:
import logging
logging.getLogger('phonemizer').setLevel(logging.ERROR)

In [4]:
import os
import glob
import random
import soundfile as sf
import numpy as np
from kokoro import KPipeline

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Replace these with the exact paths to your folders in Google Drive
INPUT_FOLDER = "/content/drive/MyDrive/Podcast_Transcripts"
OUTPUT_FOLDER = "/content/drive/MyDrive/Podcast_Audio_Files"

# A mix of the best American English Kokoro voices
# (am = American Male, af = American Female)
VOICES = [
    # --- American English (Female) ---
    'af_sky',     # Natural & friendly (comparable to OpenAI's Sky)
    'af_bella',   # Expressive & highly rated
    'af_sarah',   # Warm, professional & clear long-form reader
    'af_nicole',  # Authoritative but approachable
    'af_alloy',   # Smooth & relaxed

    # --- American English (Male) ---
    'am_adam',    # Clear, warm & conversational
    'am_michael', # Deep, authoritative podcast/audiobook style
    'am_eric',    # Upbeat, friendly podcast co-host vibe
    'am_fenrir',  # Deep, gritty, and dynamic storytelling
    'am_onyx',    # Smooth and authoritative

    # --- British English ---
    'bf_emma',    # Professional & articulate British female
    'bf_isabella',# Soft, polite storytelling British female
    'bm_george',  # Warm, trustworthy British male
    'bm_lewis'    # Crisp, professional British broadcaster
]

# Keeps Colab's RAM safe by processing 50 lines at a time before stitching
LINES_PER_CHUNK = 500

# ==========================================
# 2. GENERATION LOGIC
# ==========================================
def generate_batch_audio():
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    print("Initializing Kokoro Pipeline...")
    pipeline = KPipeline(lang_code='a')

    # Find all .txt files in your input folder
    search_pattern = os.path.join(INPUT_FOLDER, "*.txt")
    text_files = glob.glob(search_pattern)

    if len(text_files) == 0:
        print(f"Error: No text files found in {INPUT_FOLDER}")
        return

    print(f"Found {len(text_files)} transcripts. Starting batch processing...\n")

    # Loop through every text file in the folder
    for current_file_num, file_path in enumerate(text_files, 1):

        # Extract the base name (e.g., "Episode_1" from "Episode_1.txt")
        base_name = os.path.splitext(os.path.basename(file_path))[0]
        output_file = os.path.join(OUTPUT_FOLDER, f"{base_name}.wav")

        # CRITICAL: Skip if already generated.
        # This allows you to resume the script if Colab times out!
        if os.path.exists(output_file):
            print(f"[{current_file_num}/{len(text_files)}] '{base_name}.wav' already exists. Skipping...")
            continue

        # Pick a random voice for this specific file
        selected_voice = random.choice(VOICES)
        print(f"[{current_file_num}/{len(text_files)}] Processing '{base_name}' with voice: {selected_voice}")

        with open(file_path, "r", encoding="utf-8") as file:
            lines = file.readlines()

        lines = [line.strip() for line in lines if line.strip()]

        # Array to hold all the audio chunks for this specific file
        full_file_audio = []

        # Process the text in safe chunks
        for i in range(0, len(lines), LINES_PER_CHUNK):
            chunk_text = " ".join(lines[i:i + LINES_PER_CHUNK])

            try:
                # Generate audio for the chunk
                generator = pipeline(chunk_text, voice=selected_voice, speed=1.0)

                for _, _, audio in generator:
                    full_file_audio.append(audio)

            except Exception as e:
                print(f"  -> Error on a chunk in {base_name}. Skipping chunk. Details: {e}")

        # Stitch all the chunks together and save as ONE .wav file
        if full_file_audio:
            print(f"  -> Saving {output_file}...")
            final_audio = np.concatenate(full_file_audio)
            sf.write(output_file, final_audio, 24000)
            print(f"  -> ✅ Success!")
        else:
            print(f"  -> ❌ Failed to generate any audio for {base_name}.")

    print("\n🎉 Batch generation complete! All files saved to your Drive.")

# ==========================================
# 3. RUN THE SCRIPT
# ==========================================
generate_batch_audio()

Initializing Kokoro Pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

Error: No text files found in /content/drive/MyDrive/Podcast_Transcripts
